In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import ParameterGrid, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, GRU, Dense, Dropout, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import tensorflow.keras.backend as K
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import os
import tensorflow as tf

tf.autograph.set_verbosity(0)  # Tắt autograph tracing

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [2]:
# Đọc dữ liệu
df = pd.read_csv('dataset_LOSO/train.csv')

# Tách tập Test với SUB_ID = 9
test_df = df[df['SUB_ID'] == 9]
train_val_df = df[df['SUB_ID'] != 9]

In [3]:

# Định nghĩa lớp Attention
@tf.keras.utils.register_keras_serializable(package='Custom', name='Attention')
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        output = x * a
        return K.sum(output, axis=1)

# Hàm build mô hình Bi-GRU + Attention với tham số động
def build_bi_gru_attention_model(input_shape, num_classes, gru_units=64, dropout_rate=0.5, learning_rate=0.001):
    input_layer = Input(shape=input_shape)

    gru_layer = Bidirectional(GRU(gru_units, return_sequences=True, kernel_regularizer='l2'))(input_layer)
    attention_layer = Attention()(gru_layer)

    dropout_layer = Dropout(dropout_rate)(attention_layer)
    output_layer = Dense(num_classes, activation='softmax')(dropout_layer)

    model = Model(inputs=input_layer, outputs=output_layer)
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

In [4]:
some_threshold = 0.4  # Ngưỡng cho val_loss

# Hàm Grid Search
def grid_search(X_train, y_train, X_val, y_val, input_shape, num_classes, class_weights_dict):
    param_grid = {
        'gru_units': [32, 64, 128],
        'learning_rate': [0.001, 0.0005, 0.0001],
        'dropout_rate': [0.3, 0.5, 0.7]
    }

    best_params = None
    best_val_accuracy = 0

    for params in ParameterGrid(param_grid):
        
        model = build_bi_gru_attention_model(
            input_shape=input_shape,
            num_classes=num_classes,
            gru_units=params['gru_units'],
            dropout_rate=params['dropout_rate'],
            learning_rate=params['learning_rate']
        )

        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=1000,
            batch_size=32,
            verbose=0,
            class_weight=class_weights_dict,
            callbacks=[early_stopping]
        )

        val_accuracy = max(history.history['val_accuracy'])
        
        val_accuracy = max(history.history['val_accuracy'])
        val_loss = min(history.history['val_loss'])
        if val_accuracy > best_val_accuracy and val_loss < some_threshold:  # Kết hợp cả hai
            best_val_accuracy = val_accuracy
            best_params = params

    print(f"Best parameters: {best_params}, Best validation accuracy: {best_val_accuracy}, Best validation loss: {val_loss}")
    return best_params

In [5]:
subjects = train_val_df['SUB_ID'].unique()
num_classes = len(df['label'].unique())

print(f"Number of classes: {num_classes}")
print(f"Number of subjects: {len(subjects)}")

Number of classes: 6
Number of subjects: 9


In [6]:
# LOSO loop (Train/Validation: 8/1)
all_best_params = []

encoder = OneHotEncoder(sparse_output=False)

for val_subject in subjects:
    print(f'\n===== Validation on subject: {val_subject} =====')

    # Tách dữ liệu: 8 người Train, 1 người Validation
    val_df = train_val_df[train_val_df['SUB_ID'] == val_subject]
    train_df = train_val_df[train_val_df['SUB_ID'] != val_subject]

    X_train = train_df.drop(['SUB_ID', 'label'], axis=1).values
    y_train = train_df['label'].values
    X_val = val_df.drop(['SUB_ID', 'label'], axis=1).values
    y_val = val_df['label'].values

    # Chuẩn hóa
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # Định hình lại cho GRU
    X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_val = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))

    # One-hot encoding cho nhãn
    y_train = encoder.fit_transform(y_train.reshape(-1, 1))
    y_val = encoder.transform(y_val.reshape(-1, 1))

    # Tính class weights
    y_train_labels = np.argmax(y_train, axis=1)
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    class_weights_dict = dict(zip(np.unique(y_train_labels), class_weights))

    # Grid Search để tìm tham số tối ưu
    best_params = grid_search(
        X_train, y_train, X_val, y_val,
        input_shape=(X_train.shape[1], X_train.shape[2]),
        num_classes=num_classes,
        class_weights_dict=class_weights_dict
    )
    all_best_params.append(best_params)

# Chọn tham số tối ưu từ lần lặp cuối cùng
final_best_params = all_best_params[-1]
print("\nFinal best parameters after LOSO:", final_best_params)


===== Validation on subject: 0 =====

Best parameters: {'dropout_rate': 0.7, 'gru_units': 32, 'learning_rate': 0.001}, Best validation accuracy: 0.9418604373931885, Best validation loss: 0.3083854913711548

===== Validation on subject: 1 =====
Best parameters: {'dropout_rate': 0.5, 'gru_units': 128, 'learning_rate': 0.0005}, Best validation accuracy: 0.9474071264266968, Best validation loss: 0.3010520040988922

===== Validation on subject: 2 =====
Best parameters: {'dropout_rate': 0.3, 'gru_units': 32, 'learning_rate': 0.001}, Best validation accuracy: 0.9449609518051147, Best validation loss: 0.2749461829662323

===== Validation on subject: 3 =====
Best parameters: {'dropout_rate': 0.5, 'gru_units': 128, 'learning_rate': 0.001}, Best validation accuracy: 0.9927272796630859, Best validation loss: 0.2745796740055084

===== Validation on subject: 4 =====
Best parameters: None, Best validation accuracy: 0, Best validation loss: 0.7987699508666992

===== Validation on subject: 5 =====
Bes

In [7]:
# Chuẩn bị dữ liệu toàn bộ tập Train (9 người)
X_train_full = train_val_df.drop(['SUB_ID', 'label'], axis=1).values
y_train_full = train_val_df['label'].values
X_test = test_df.drop(['SUB_ID', 'label'], axis=1).values
y_test = test_df['label'].values

# Chuẩn hóa
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# Định hình lại cho GRU
X_train_full = X_train_full.reshape((X_train_full.shape[0], 1, X_train_full.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# One-hot encoding cho nhãn
y_train_full = encoder.fit_transform(y_train_full.reshape(-1, 1))
y_test = encoder.transform(y_test.reshape(-1, 1))

# Tính class weights cho toàn bộ tập Train
y_train_full_labels = np.argmax(y_train_full, axis=1)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_full_labels), y=y_train_full_labels)
class_weights_dict = dict(zip(np.unique(y_train_full_labels), class_weights))

# Huấn luyện với cross-validation (8/1) sử dụng tham số tối ưu
kfold = KFold(n_splits=9, shuffle=True, random_state=42)
best_model = None
best_val_loss = float('inf')

for train_idx, val_idx in kfold.split(X_train_full):
    # Kiểm tra nếu final_best_params là None thì bỏ qua vòng lặp này
    if final_best_params is None:
        print("final_best_params là None, bỏ qua vòng lặp này...")
        continue

    X_train_cv = X_train_full[train_idx]
    y_train_cv = y_train_full[train_idx]
    X_val_cv = X_train_full[val_idx]
    y_val_cv = y_train_full[val_idx]

    # Build model với tham số tối ưu
    model = build_bi_gru_attention_model(
        input_shape=(X_train_full.shape[1], X_train_full.shape[2]),
        num_classes=num_classes,
        gru_units=final_best_params['gru_units'],
        dropout_rate=final_best_params['dropout_rate'],
        learning_rate=final_best_params['learning_rate']
    )

    # Callbacks với Early Stopping
    checkpoint = ModelCheckpoint("Model/Squat_detection_GRU_LOSO.keras", save_best_only=True, monitor="val_loss", mode="min", verbose=0)
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)

    # Huấn luyện
    history = model.fit(
        X_train_cv, y_train_cv,
        validation_data=(X_val_cv, y_val_cv),
        epochs=50,
        batch_size=32,
        verbose=0,
        class_weight=class_weights_dict,
        callbacks=[checkpoint, early_stopping]
    )

    # Cập nhật mô hình tốt nhất
    val_loss = min(history.history['val_loss'])
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model

In [12]:
# Đánh giá và in accuracy, loss, và validation loss của mô hình tốt nhất
if best_model is not None:
    test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"Best Model - Validation Loss: {best_val_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
else:
    print("No best model found. Please check the training process.")

Best Model - Validation Loss: 0.0679, Test Loss: 0.1144, Test Accuracy: 0.9724


In [13]:
# Dự đoán và đánh giá trên tập Test (SUB_ID = 9)
y_pred = best_model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Tính test loss và test accuracy
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)

# Tạo classification report
report = classification_report(y_true_classes, y_pred_classes, output_dict=True)

# In kết quả
print("\n=== Final Evaluation for Test (SUB_ID=9) ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(pd.DataFrame(report).T)


=== Final Evaluation for Test (SUB_ID=9) ===
Test Loss: 0.1144
Test Accuracy: 0.9724

Classification Report:
              precision    recall  f1-score      support
0              0.944588  0.960682  0.952567   763.000000
1              1.000000  1.000000  1.000000   382.000000
2              0.996974  1.000000  0.998485   659.000000
3              1.000000  0.975709  0.987705   247.000000
4              0.849246  0.820388  0.834568   206.000000
5              1.000000  0.995690  0.997840   464.000000
accuracy       0.972437  0.972437  0.972437     0.972437
macro avg      0.965135  0.958745  0.961861  2721.000000
weighted avg   0.972316  0.972437  0.972323  2721.000000


In [10]:
import joblib

# Lưu scaler ra file
joblib.dump(scaler, 'Model/scaler_GRU_LOSO.pkl')

['Model/scaler_GRU_LOSO.pkl']